In [441]:
import numpy as np
import scipy as sp
import sympy as smp
from sympy import kronecker_product as kron
from sympy import init_printing, print_latex
import random

The equations for $Ann_\mathfrak{g}(\psi \otimes 0)$ are: $$ \alpha a + \beta b = - \alpha e \\ \alpha c - \beta a = - \beta e \\ f = 0 \\ d \in \mathbb{C}^2 $$

In [442]:
α, β = smp.symbols("α β")

# basis for sl2
e = smp.Matrix([ # raise
    [0, 1],
    [0, 0]
])
f = smp.Matrix([ # lower
    [0, 0],
    [1, 0]
])
h = smp.Matrix([ # cartan
    [1, 0],
    [0, -1]
])

I = smp.eye(2)
Z = smp.zeros(2)

# basis for sl2 + sl2
# pick the order so that sympy uses the same free variables I chose (last 3 are cHOSEN TO BE FREE)

basis = [
    (f, Z),   # c  -> determined
    (Z, h),   # e  -> determined
    (Z, f),   # f  -> determined
    (h, Z),   # a  -> free
    (e, Z),   # b  -> free
    (Z, e),   # d  -> free
]
 
def rho(A, B): # rep of (a, b) for acting on C2 x C2
    return kron(A, I) + kron(I, B)

sl2_2_basis = [rho(A,B) for (A, B) in basis]

psi = smp.Matrix([
    [α],
    [β]
])
ket0 = smp.Matrix([
    [1],
    [0]
])

ket_psi_0 = kron(psi, ket0)

display(len(sl2_2_basis))
display(sl2_2_basis[0])
display(kron(I, e))

6

⎡0  0  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢1  0  0  0⎥
⎢          ⎥
⎣0  1  0  0⎦

⎡0  1  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  1⎥
⎢          ⎥
⎣0  0  0  0⎦

In [443]:
# get the matrix of all actionsso 
M = smp.Matrix.hstack(*[g * ket_psi_0 for g in sl2_2_basis])
M

# with the two qubit basis |00>, |01>, |10>, |11>, the columns are 

⎡0  α  0  α   β  0⎤
⎢                 ⎥
⎢0  0  α  0   0  0⎥
⎢                 ⎥
⎢α  β  0  -β  0  0⎥
⎢                 ⎥
⎣0  0  β  0   0  0⎦

In [444]:
# take the nullspace of this matrix:

nullspace = M.nullspace()

for m in nullspace:
    display(m)

⎡2⋅β⎤
⎢───⎥
⎢ α ⎥
⎢   ⎥
⎢-1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎣ 0 ⎦

⎡ 2 ⎤
⎢β  ⎥
⎢── ⎥
⎢ 2 ⎥
⎢α  ⎥
⎢   ⎥
⎢-β ⎥
⎢───⎥
⎢ α ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 1 ⎥
⎢   ⎥
⎣ 0 ⎦

⎡0⎤
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎢0⎥
⎢ ⎥
⎣1⎦

In [445]:
M.rank(), len(nullspace)

(3, 3)

In [446]:
# the nullspace of M is just vectors Mv = 0, where v = [c, e, f, a, b, d], aka a chosen basis for Ann_g()
# reshape the matrices to get the proper basis
ann_g_basis = []

for k, n in enumerate(nullspace):
    A = smp.simplify(sum((n[i] * basis[i][0] for i in range(6)), smp.zeros(2)))
    B = smp.simplify(sum((n[i] * basis[i][1] for i in range(6)), smp.zeros(2)))
    
    print("coefficient vector:")
    display(n)
    print(f'\nbasis vector {k+1}:')
    display((A, B))
    rep = kron(A, I) + kron(I, B)
    display(rep)
    ann_g_basis.append(rep)

random_c = lambda: random.random() + 1j * random.random()
def random_c2():
    vec = [random_c(), random_c()]
    return vec / np.linalg.norm(vec)

print(f"Random psi in C2: \n{random_c2()}")

# show that for a random vector in the annihilator, it always annihilates
rep = sum(random_c() * np.array(X.tolist()) for X in ann_g_basis)

print(f"Random matrix in Ann_g(|ψ0>): {rep}")

# evaluate for a random |ψ>
(rep @ ket_psi_0).subs({α: random_c2()[0], β: random_c2()[1]}).evalf(2)

coefficient vector:


⎡2⋅β⎤
⎢───⎥
⎢ α ⎥
⎢   ⎥
⎢-1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎢ 1 ⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎣ 0 ⎦


basis vector 1:


⎛⎡ 1   0 ⎤         ⎞
⎜⎢       ⎥  ⎡-1  0⎤⎟
⎜⎢2⋅β    ⎥, ⎢     ⎥⎟
⎜⎢───  -1⎥  ⎣0   1⎦⎟
⎝⎣ α     ⎦         ⎠

⎡ 0    0   0   0⎤
⎢               ⎥
⎢ 0    2   0   0⎥
⎢               ⎥
⎢2⋅β            ⎥
⎢───   0   -2  0⎥
⎢ α             ⎥
⎢               ⎥
⎢     2⋅β       ⎥
⎢ 0   ───  0   0⎥
⎣      α        ⎦

...


basis vector 3:


⎛⎡0  0⎤  ⎡0  1⎤⎞
⎜⎢    ⎥, ⎢    ⎥⎟
⎝⎣0  0⎦  ⎣0  0⎦⎠

⎡0  1  0  0⎤
⎢          ⎥
⎢0  0  0  0⎥
⎢          ⎥
⎢0  0  0  1⎥
⎢          ⎥
⎣0  0  0  0⎦

Random psi in C2: 
[0.55705503+0.65917592j 0.49542333+0.09865356j]
Random matrix in Ann_g(|ψ0>): [[-β*(0.601675188420316 + 0.15454173026646*I)/α
  0.291090135952814 + 0.622098019381273*I
  0.601675188420316 + 0.15454173026646*I 0]
 [0
  1.67598883350797 + 1.78440751893359*I + β*(0.601675188420316 + 0.15454173026646*I)/α
  0 0.601675188420316 + 0.15454173026646*I]
 [2*β*(0.837994416753987 + 0.892203759466795*I)/α + β**2*(0.601675188420316 + 0.15454173026646*I)/α**2
  0
  -1.67598883350797 - 1.78440751893359*I - β*(0.601675188420316 + 0.15454173026646*I)/α
  0.291090135952814 + 0.622098019381273*I]
 [0
  2*β*(0.837994416753987 + 0.892203759466795*I)/α + β**2*(0.601675188420316 + 0.15454173026646*I)/α**2
  0 β*(0.601675188420316 + 0.15454173026646*I)/α]]


⎡         0          ⎤
⎢                    ⎥
⎢         0          ⎥
⎢                    ⎥
⎢-4.4e-17 - 1.4e-17⋅ⅈ⎥
⎢                    ⎥
⎣         0          ⎦

In [447]:
# now lets move onto computing the complement of this annihilator to get the subalgebra that doesn't take the vector to 0

# rank = smp.Matrix.hstack(*nullspace)
# # display(np.array(rank.tolist()))
# np_nullspace = np.array(rank.tolist())
# print(np_nullspace)

# np.array(rho(Z, f).tolist()).flatten()
# display(rho(Z, f))
ann_g_basis.append(rho(Z, f))
ann_g_basis = [a.subs({α: (1/np.sqrt(2)), β: (1/np.sqrt(2))}).evalf() for a in ann_g_basis]

# flatten
new = [smp.Matrix(list(a.T)) for a in ann_g_basis]
display(new)

rank_m = smp.Matrix.hstack(*new)
display(rank_m)

rank_m.rank()

⎡⎡ 0  ⎤  ⎡-1.0⎤  ⎡ 0 ⎤  ⎡ 0 ⎤⎤
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢1.0⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢2.0 ⎥  ⎢1.0 ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢1.0⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢2.0 ⎥  ⎢1.0 ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢2.0 ⎥  ⎢1.0 ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥, ⎢    ⎥, ⎢   ⎥, ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢1.0 ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢-2.0⎥  ⎢-1.0⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢1.0⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢1.0 ⎥  ⎢ 0 ⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎢⎢ 0  ⎥  ⎢ 0  ⎥  ⎢1.0⎥  ⎢ 0 ⎥⎥
⎢⎢    ⎥  ⎢    ⎥  ⎢   ⎥  ⎢   ⎥⎥
⎣⎣ 0  ⎦  ⎣1.0 ⎦  ⎣ 0 ⎦  ⎣ 0 ⎦⎦

⎡ 0    -1.0   0    0 ⎤
⎢                    ⎥
⎢ 0     0     0   1.0⎥
⎢                    ⎥
⎢2.0   1.0    0    0 ⎥
⎢                    ⎥
⎢ 0     0     0    0 ⎥
⎢                    ⎥
⎢ 0     0    1.0   0 ⎥
⎢                    ⎥
⎢2.0   1.0    0    0 ⎥
⎢                    ⎥
⎢ 0     0     0    0 ⎥
⎢                    ⎥
⎢2.0   1.0    0    0 ⎥
⎢                    ⎥
⎢ 0    1.0    0    0 ⎥
⎢                    ⎥
⎢ 0     0     0    0 ⎥
⎢                    ⎥
⎢-2.0  -1.0   0    0 ⎥
⎢                    ⎥
⎢ 0     0     0   1.0⎥
⎢                    ⎥
⎢ 0     0     0    0 ⎥
⎢                    ⎥
⎢ 0    1.0    0    0 ⎥
⎢                    ⎥
⎢ 0     0    1.0   0 ⎥
⎢                    ⎥
⎣ 0    1.0    0    0 ⎦

4

In [ ]:
# boom just found a 4th basis vector